# Batch Inference
Goal: Load new daily data, apply ML transformations, predict, and save results

In [0]:
import mlflow
import pandas as pd

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

## Load Cleaned Test Dataset (New Daily Data)

In [0]:
print("🔄 Starting Batch Inference Process...")

# Load Cleaned Test Dataset (New Daily Data)
ml_test_dataset = spark.table("workspace.insurance_claim.ml_test_dataset")

## Data Preparation (Feature Assembly)

In [0]:
# Data Preparation (Feature Assembly)

# Define ignore columns exactly as in the training phase to prevent schema mismatch
ignore_cols = [
    "ClaimID", "BeneID", "Provider", "is_fraud",
    "AttendingPhysician", "OperatingPhysician", "OtherPhysician",
    "ClmDiagnosisCode_2", "ClmDiagnosisCode_3", "ClmDiagnosisCode_4", 
    "ClmDiagnosisCode_5", "ClmDiagnosisCode_6", "ClmDiagnosisCode_7", 
    "ClmDiagnosisCode_8", "ClmDiagnosisCode_9", "ClmDiagnosisCode_10",
    "ClmProcedureCode_1", "ClmProcedureCode_2", "ClmProcedureCode_3", 
    "ClmProcedureCode_4", "ClmProcedureCode_5", "ClmProcedureCode_6",
    "ClaimStartDt", "ClaimEndDt", "AdmissionDt", "DischargeDt", "DOB", "DOD"
]

categorical_cols = [f.name for f in ml_test_dataset.schema.fields if isinstance(f.dataType, StringType) and f.name not in ignore_cols]

numeric_cols = [f.name for f in ml_test_dataset.schema.fields if not isinstance(f.dataType, StringType) and f.name not in ignore_cols]

indexed_cat_cols = [f"{c}_indexed" for c in categorical_cols]

print("⚙️ Assembling features...")

# Rebuild indexing and assembling logic to create the 'features' vector
indexer = StringIndexer(inputCols=categorical_cols, outputCols=indexed_cat_cols, handleInvalid="keep")
indexed_df = indexer.fit(ml_test_dataset).transform(ml_test_dataset)

assembler_inputs = numeric_cols + indexed_cat_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")
prepared_test_df = assembler.transform(indexed_df)

print("✅ 'features' column created successfully!")

## Load Registered Model from Unity Catalog

In [0]:
# Load Registered Model from Unity Catalog

model_name = "workspace.insurance_claim.fraud_gbt_model"
model_version = 2  # 🌟 Ensure this version number matches your latest registered model
model_uri = f"models:/{model_name}/{model_version}"
tmp_vol_path = "/Volumes/workspace/insurance_claim/mlflow_tmp"

print(f"📥 Loading GBT Model from: {model_uri}")
loaded_gbt_model = mlflow.spark.load_model(
    model_uri=model_uri,
    dfs_tmpdir=tmp_vol_path
)

## Generate Predictions & Save to Catalog

In [0]:
# Generate Predictions & Save to Catalog

print("🎯 Generating fraud predictions...")
predictions = loaded_gbt_model.transform(prepared_test_df)

# Extract only the essential columns required by the business/investigation team
final_results = predictions.select(
    col("Provider"), 
    col("prediction").alias("predicted_fraud"), 
    col("probability")
)

# Save the final prediction results as a new table in Unity Catalog
output_table = "workspace.insurance_claim.daily_fraud_predictions"
final_results.write.mode("overwrite").saveAsTable(output_table)

print("✅ Batch inference completed!")
print(f"💾 Saved predictions to: {output_table}")

In [0]:
# Display the results for quick inspection
display(final_results)

## Saved result to Catalog

In [0]:
# Saved result to Catalog

output_table = "workspace.insurance_claim.daily_fraud_predictions"
final_results.write.mode("overwrite").saveAsTable(output_table)
print(f"💾 Saved predictions to: {output_table}")

In [0]:
# Extract Feature Importances from GBT Model
importances = loaded_gbt_model.stages[-1].featureImportances.toArray()

# Extract all columns from VectorAssembler
feature_names = assembler_inputs

# Gather into DataFrame for ranking
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Display top 10 columns which effect to fraud detection
print("🏆 Top 10 Most Important Features:")
print(importance_df.head(10))

%md
### 🔍 Model Interpretation

**1. ClmDiagnosisCode_1_indexed (52.4%)**

Definition: 

The primary diagnosis code is the number one variable used by the model to detect fraud.Business Perspective: Diagnosis patterns are the strongest indicators of fraud. For example, specific disease codes may be recorded with abnormal frequency solely to maximize insurance reimbursements.

**2. County_indexed (23.7%) & State_indexed (11.8%)**

Definition: 

Location-based variables (County and State) collectively account for approximately 35.5% of the model's decision-making weight.

Business Perspective: 

Fraudulent activities tend to form geographical clusters. This points to coordinated fraud networks involving specific healthcare facilities or provider groups operating within certain areas.

**3. ClmAdmitDiagnosisCode_indexed (8.5%) & DiagnosisGroupCode_indexed (3.5%)**

Definition: The admitting diagnosis code and the Diagnosis-Related Group (DRG) code capture nearly all the remaining model weight.


**⚠️ Key Observation:**

Zero Importance Variables (0.000000)

You will notice that numerical features such as TotalClaimCost, DeductibleRatio, and Is_Inpatient hold a feature importance of exactly 0.000000. This occurs due to two main reasons:

1. Tree Depth & Model Sufficiency: 

The Gradient Boosted Trees (GBT) model achieves near-perfect split decisions within the first 5 to 6 categorical columns (which consume ~100% of the weight). Consequently, the model does not need to utilize the remaining columns to make a definitive classification.

2. Categorical Dominance: 

In this particular dataset, medical diagnosis codes and geographic locations hold significantly higher predictive power and distinct signal dominance compared to monetary values.

**💡 Next Steps & Actionable Insights**

1. Executive & Investigative Briefing (Explainability): 

You can confidently present to stakeholders that "This fraud detection system does not merely flag high-dollar claims. Instead, it systematically identifies high-risk behavior based on clinical diagnosis combinations and localized geographical anomalies."

2. Feature Selection for Optimization: 

To streamline the data pipeline and improve compute efficiency in future iterations, columns with 0.0 importance can be safely dropped. This will minimize memory consumption and accelerate total training time.